In [4]:
import json
import torch
import pickle
import functools

def load_jsonlines(fname: str):
    """Read jsonlines file."""
    with open(fname, "r") as f:
        return [json.loads(line) for line in f]


def dump_jsonlines(obj, fname: str, indent: int = None):
    """Dump jsonlines file."""
    with open(fname, "w", encoding="utf-8") as outfile:
        for entry in obj:
            json.dump(entry, outfile, indent=indent)
            outfile.write("\n")


def load_json(fname: str):
    """Read json file."""
    with open(fname, "r") as f:
        return json.load(f)


def dump_json(obj, fname: str, indent: int = None):
    """Dump json file."""
    with open(fname, "w", encoding="utf-8") as f:
        return json.dump(obj, f, indent=indent)

In [17]:
# cre_train = io.load_jsonlines("/u/zliu/datastor1/KE-by-CP/data/debug_meta_train/syn_data_neurips/4Ktrain_data_100percent_frozen/train_text_data_id_entity152_rel31.jsonl")
# cre_train = io.load_jsonlines("/u/zliu/datastor1/KE-by-CP/data/debug_meta_train/syn_data_neurips/4Ktrain_data_100percent_frozen/valid_text_data_id_entity152_rel31.jsonl")
cre_train = load_jsonlines("/home1/09636/zyliu/work/RLEdit/data/raw/4Ktrain_data_100percent_frozen/test_text_data_ood-relation_entity152_rel7.jsonl")

In [18]:
# loc_train = io.load_json("/u/zliu/datastor1/RLEdit/data/raw/zsre/zsre_train.json")
loc_train = load_json("/home1/09636/zyliu/work/RLEdit/data/raw/zsre/zsre_eval.json")

In [19]:
loc_train[0]

{'subject': 'Watts Humphrey',
 'src': 'What university did Watts Humphrey attend?',
 'rephrase': 'What university did Watts Humphrey take part in?',
 'alt': 'University of Michigan',
 'loc': 'nq question: who played desmond doss father in hacksaw ridge',
 'loc_ans': 'Hugo Weaving',
 'ans': 'Illinois Institute of Technology'}

In [15]:
new_cre_train = []
for i in range(len(cre_train)):
    cre_d = cre_train[i]
    new_cre_train.append(
        {
            "text": cre_d["text"],
            "questions": [q["alias_question"] for q in cre_d["questions"]],
            "answers": [str(q["answer"]) for q in cre_d["questions"]],
            "loc": loc_train[i]["loc"],
            "loc_ans": loc_train[i]["loc_ans"],
        }
    )

In [20]:
dump_json(new_cre_train, "/home1/09636/zyliu/work/RLEdit/data/raw/ctrlRE/test_ood_relation.json")
# io.dump_json(new_cre_train, "/u/zliu/datastor1/RLEdit/data/raw/ctrlRE/eval_4K.json")

In [ ]:
import sys
# sys.path.append("/u/zliu/datastor1/RLEdit")
from cre import CREDataset

ImportError: attempted relative import with no known parent package

In [97]:
head_dim=8
theta_base = 10000
seq_len = 10

# (seq_len, head_dim/2
# [m_1*theta_1 m_1*theta_2 ... m_1*theta_head_dim/2]
# ...

def precompute_inverse_frequency_matrix(dim, theta_base, max_seq_len: int = 4096) -> None:
    theta = 1.0 / (theta_base ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    
    # Create position indexes `[0, 1, ..., max_seq_len - 1]`
    seq_idx = torch.arange(
        max_seq_len, dtype=theta.dtype, device=theta.device
    )

    # Outer product of theta and position index; output tensor has
    # a shape of [max_seq_len, dim // 2]
    idx_theta = torch.einsum("i, j -> ij", seq_idx, theta).float()

    # cache includes both the cos and sin components and so the output shape is
    # [max_seq_len, dim // 2, 2]
    inv_freq = torch.stack([torch.cos(idx_theta), torch.sin(idx_theta)], dim=-1)
    inv_freq = torch.repeat_interleave(inv_freq, repeats=2, dim=-2)
    inv_freq = inv_freq
    return inv_freq

def rotate_half(x): 
    x_shape = x.shape
    even_x = x[..., ::2]
    odd_x = x[..., 1::2]
    x = torch.stack([-odd_x, even_x], dim=-1).view(*x_shape[:-1], -1)
    return x

def apply_rotary_embeddings(x, inv_freq):
    # x : B, T, n_head, head_dim    
    seq_len = x.size(1)
    
    seq_inv_freq = inv_freq[:seq_len].unsqueeze(1).unsqueeze(0)
    cos, sin = seq_inv_freq.unbind(-1)
    
    return cos * x + rotate_half(x) * sin

In [98]:

x = torch.arange(8)

# B x T, n_head, head_dim
x = x.float().unsqueeze(0).unsqueeze(0).unsqueeze(0).repeat([1, 10, 1, 1])

inv_freq = precompute_inverse_frequency_matrix(dim=head_dim, theta_base=10000, max_seq_len=4096)
my_x_rotate = apply_rotary_embeddings(x, inv_freq)
my_x_rotate.shape

torch.Size([1, 10, 1, 8])

In [80]:


x = torch.arange(8)

# B x T, n_head, head_dim
x = x.float().unsqueeze(0).unsqueeze(0).unsqueeze(0).repeat([1, 10, 1, 1])
x.shape

torch.Size([1, 10, 1, 8])

In [116]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    Precompute the frequency tensor for complex exponentials (cis) with given dimensions.

    This function calculates a frequency tensor with complex exponentials using the given dimension 'dim'
    and the end index 'end'. The 'theta' parameter scales the frequencies.
    The returned tensor contains complex values in complex64 data type.

    Args:
        dim (int): Dimension of the frequency tensor.
        end (int): End index for precomputing frequencies.
        theta (float, optional): Scaling factor for frequency computation. Defaults to 10000.0.

    Returns:
        torch.Tensor: Precomputed frequency tensor with complex exponentials.

    
        

    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)  # type: ignore
    freqs = torch.outer(t, freqs).float()  # type: ignore
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    """
    Reshape frequency tensor for broadcasting it with another tensor.

    This function reshapes the frequency tensor to have the same shape as the target tensor 'x'
    for the purpose of broadcasting the frequency tensor during element-wise operations.

    Args:
        freqs_cis (torch.Tensor): Frequency tensor to be reshaped.
        x (torch.Tensor): Target tensor for broadcasting compatibility.

    Returns:
        torch.Tensor: Reshaped frequency tensor.

    Raises:
        AssertionError: If the frequency tensor doesn't match the expected shape.
        AssertionError: If the target tensor 'x' doesn't have the expected number of dimensions.
    """
    ndim = x.ndim
    assert 0 <= 1 < ndim
    assert freqs_cis.shape == (x.shape[1], x.shape[-1])
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
):
    """
    Apply rotary embeddings to input tensors using the given frequency tensor.

    This function applies rotary embeddings to the given query 'xq' and key 'xk' tensors using the provided
    frequency tensor 'freqs_cis'. The input tensors are reshaped as complex numbers, and the frequency tensor
    is reshaped for broadcasting compatibility. The resulting tensors contain rotary embeddings and are
    returned as real tensors.

    Args:
        xq (torch.Tensor): Query tensor to apply rotary embeddings.
        xk (torch.Tensor): Key tensor to apply rotary embeddings.
        freqs_cis (torch.Tensor): Precomputed frequency tensor for complex exponentials.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Tuple of modified query tensor and key tensor with rotary embeddings.

        

    """
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)


x = torch.arange(8)

# B x T, n_head, head_dim
x = x.float().unsqueeze(0).unsqueeze(0).unsqueeze(0).repeat([1, 10, 1, 1])

freq_complex = precompute_freqs_cis(dim=8, end=10)
reference_x_rotate, _ = apply_rotary_emb(x, x, freq_complex)

In [117]:
reference_x_rotate

tensor([[[[ 0.0000,  1.0000,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[-0.8415,  0.5403,  1.6905,  3.1847,  3.9498,  5.0397,  5.9930,
            7.0060]],

         [[-0.9093, -0.4161,  1.3641,  3.3375,  3.8992,  5.0790,  5.9860,
            7.0120]],

         [[-0.1411, -0.9900,  1.0241,  3.4570,  3.8482,  5.1177,  5.9790,
            7.0180]],

         [[ 0.7568, -0.6536,  0.6739,  3.5420,  3.7969,  5.1560,  5.9720,
            7.0239]],

         [[ 0.9589,  0.2837,  0.3169,  3.5916,  3.7451,  5.1937,  5.9649,
            7.0299]],

         [[ 0.2794,  0.9602, -0.0433,  3.6053,  3.6930,  5.2309,  5.9579,
            7.0359]],

         [[-0.6570,  0.7539, -0.4030,  3.5830,  3.6405,  5.2675,  5.9509,
            7.0418]],

         [[-0.9894, -0.1455, -0.7587,  3.5248,  3.5876,  5.3037,  5.9438,
            7.0478]],

         [[-0.4121, -0.9111, -1.1068,  3.4315,  3.5344,  5.3393,  5.9368,
            7.0537]]]])

In [113]:
def liyan_precompute_frequency_complex_matrix(head_dim: int, seq_len, theta: float=10000.0):

    # 1 / (10000 ^ (2i /d)); i = 0, ..., head_dim/2
    # (head_dim/2, )
    theta = 1 / (theta ** torch.arange(0, head_dim, 2).float() / head_dim)

    # (seq_len,)
    m = torch.arange(0, seq_len)

    # (seq_len, head_dim/2)
    frequency_matrix = torch.outer(m, theta)

    # (seq_len, head_dim/2)
    # [m_1*theta_1 m_1*theta_2 ... m_1*theta_head_dim/2]
    # ...
    # [m_m*theta_1 m_m*theta_2 ... m_m*theta_head_dim/2]
    freq_complex = torch.polar(torch.ones_like(frequency_matrix), frequency_matrix)

    return freq_complex


def liyan_apply_rotary_embeddings(x: torch.Tensor, freq_complex: torch.Tensor):

    # (batch_size, seq_len, H, head_dim) -> (batch_size, seq_len, H, head_dim/2, 2) -> (batch_size, seq_len, H, head_dim/2)
    x_complex = torch.view_as_complex(x.view(*x.shape[:-1], -1, 2))  # TODO: use reshape instead of view

    # (seq_len, head_dim/2) -> (1, seq_len, head_dim/2) -> (1, seq_len, 1, head_dim/2)
    freq_complex = freq_complex.unsqueeze(0).unsqueeze(2)

    # (batch_size, seq_len, H, head_dim/2) -> (batch_size, seq_len, H, head_dim/2)
    x_rotate = x_complex * freq_complex

    # (batch_size, seq_len, H, head_dim/2) -> (batch_size, seq_len, H, head_dim/2, 2)
    x_rotate = torch.view_as_real(x_rotate)

    # (batch_size, seq_len, H, head_dim/2, 2) -> (batch_size, seq_len, H, head_dim)
    x_rotate = x_rotate.reshape(*x.shape).type_as(x)

    return x_rotate


x = torch.arange(8)

# B x T, n_head, head_dim
x = x.float().unsqueeze(0).unsqueeze(0).unsqueeze(0).repeat([1, 10, 1, 1])
x.shape
freq_complex = liyan_precompute_frequency_complex_matrix(head_dim=8, seq_len=10)
liyan_x_rotate = liyan_apply_rotary_embeddings(x, freq_complex)

In [115]:
liyan_x_rotate

tensor([[[[ 0.0000,  1.0000,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[-0.9894, -0.1455,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[ 0.2879, -0.9577,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[ 0.9056,  0.4242,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[-0.5514,  0.8342,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[-0.7451, -0.6669,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[ 0.7683, -0.6401,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[ 0.5216,  0.8532,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[-0.9200,  0.3919,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]],

         [[-0.2538, -0.9673,  2.0000,  3.0000,  4.0000,  5.0000,  6.0000,
            7.0000]]]])

In [104]:
torch.allclose(my_x_rotate, reference_x_rotate)

True